# Procesamiento de lenguaje natural · Modelos de lenguaje y clasificación

Continuación de *Texto, frecuencias y TF-IDF*: un modelo de lenguaje de bigramas con suavizado de Laplace y aditivo, la perplejidad, la generación de frases con una cadena de Markov, un clasificador Naive Bayes multinomial hecho a mano frente a `MultinomialNB`, un análisis de sentimiento por léxico con negaciones y un corrector ortográfico al estilo de [Peter Norvig](https://norvig.com/spell-correct.html). Los textos y las opiniones son inventados y se escriben en el propio notebook.

In [1]:
import math
import random
import re
from collections import Counter, defaultdict
from itertools import pairwise

import numpy as np

FRASES = """
la guagua sale de la estación a las ocho
la guagua llega a la playa a las nueve
la guagua de la tarde sale a las cinco
el barco sale del puerto a las siete
el barco llega a la gomera a las nueve
el barco de la tarde llega a las seis
mi abuela prepara gofio por la mañana
mi abuela prepara potaje los domingos
mi tío prepara papas con mojo los domingos
los domingos vamos a la playa
los sábados vamos al mercado
en verano vamos a la playa de las teresitas
en invierno subimos al teide
el teide tiene nieve en invierno
la playa tiene arena negra
la playa de la tarde tiene mucho viento
en la playa hace calor en verano
mi tío pesca en el puerto los sábados
""".strip().splitlines()

PRUEBA = [
    "la guagua sale del puerto a las seis de la tarde",
    "mi abuela prepara papas los sábados",
    "el barco de la mañana llega a la playa",
]


def tokens(frase):
    return re.findall(r"[a-záéíóúüñ]+", frase.lower())


def con_marcas(frase):
    return ["<s>", *tokens(frase), "</s>"]


print(len(FRASES), "frases de entrenamiento,", len(PRUEBA), "de prueba")

18 frases de entrenamiento, 3 de prueba


## 1. Recuentos de bigramas

Añadir a cada frase las marcas `<s>` (inicio) y `</s>` (fin), y contar unigramas y bigramas. Mostrar el tamaño del vocabulario (con `</s>`, sin `<s>`, que nunca se predice), los cinco bigramas más frecuentes y las palabras que se han visto detrás de `sale`. Comprobar que, para cada palabra, la suma de los recuentos de los bigramas que empiezan por ella es su número de apariciones como contexto.

In [2]:
unigramas = Counter(w for f in FRASES for w in con_marcas(f))
bigramas = Counter(par for f in FRASES for par in pairwise(con_marcas(f)))
siguientes = defaultdict(Counter)
for (a, b), c in bigramas.items():
    siguientes[a][b] += c

VOCAB = sorted(set(unigramas) - {"<s>"})
V = len(VOCAB)
print("Vocabulario:", V, "tipos")
print("Más frecuentes:", bigramas.most_common(5))
print("Detrás de «sale»:", dict(siguientes["sale"]))

for a in siguientes:
    assert sum(siguientes[a].values()) == unigramas[a] - (a == "</s>")
assert unigramas["<s>"] == len(FRASES)

Vocabulario: 53 tipos
Más frecuentes: [(('a', 'las'), 6), (('la', 'playa'), 6), (('<s>', 'la'), 5), (('de', 'la'), 4), (('a', 'la'), 4)]
Detrás de «sale»: {'de': 1, 'a': 1, 'del': 1}


## 2. Probabilidades con suavizado

La estimación de máxima verosimilitud $P(b \mid a) = \frac{c(a, b)}{c(a)}$ da probabilidad 0 a cualquier bigrama no visto. El suavizado aditivo suma $k$ a cada recuento: $P_k(b \mid a) = \frac{c(a, b) + k}{c(a) + kV}$, y con $k = 1$ es el suavizado de Laplace. Escribir `prob(a, b, k)` y comprobar que, para cada contexto y cada $k$, las probabilidades de las $V$ palabras siguientes suman 1. Comparar $P(\text{llega} \mid \text{guagua})$ y $P(\text{nieve} \mid \text{guagua})$ con distintos $k$.

In [3]:
def prob(a, b, k=1.0):
    return (bigramas[a, b] + k) / (unigramas[a] + k * V)


print(f"{'k':>6} {'P(llega|guagua)':>16} {'P(nieve|guagua)':>16}")
for k in (0, 0.01, 0.1, 1):
    print(f"{k:>6} {prob('guagua', 'llega', k):>16.4f} {prob('guagua', 'nieve', k):>16.4f}")

for k in (0.01, 0.5, 1, 3):
    for a in ["<s>", *VOCAB]:
        if a != "</s>":
            assert math.isclose(sum(prob(a, b, k) for b in VOCAB), 1)

     k  P(llega|guagua)  P(nieve|guagua)
     0           0.3333           0.0000
  0.01           0.2861           0.0028
   0.1           0.1325           0.0120
     1           0.0357           0.0179


## 3. Probabilidad de una frase y perplejidad

La probabilidad de una frase es el producto de las probabilidades de sus bigramas, incluidos los de `<s>` y `</s>`. La perplejidad es $PP = P(w_1 \dots w_N)^{-1/N}$, con $N$ el número de bigramas: cuanto menor, mejor predice el modelo. Calcularla sumando logaritmos (para no perder precisión) y comprobarla con el producto directo de `math.prod`. Comparar las frases de prueba con su versión desordenada y mostrar qué pasa con la máxima verosimilitud ($k = 0$) cuando aparece un bigrama nuevo.

In [4]:
def log_prob(frase, k=1.0):
    total = 0.0
    for a, b in pairwise(con_marcas(frase)):
        p = prob(a, b, k)
        if p == 0:
            return -math.inf
        total += math.log(p)
    return total


def perplejidad(frase, k=1.0):
    n = len(con_marcas(frase)) - 1
    return math.exp(-log_prob(frase, k) / n)


def perplejidad_producto(frase, k=1.0):
    marcas = con_marcas(frase)
    p = math.prod(prob(a, b, k) for a, b in pairwise(marcas))
    return p ** (-1 / (len(marcas) - 1)) if p else math.inf


rng = random.Random(36)
print(f"{'frase':<50} {'PP':>7} {'desordenada':>12} {'PP k=0':>8}")
for frase in PRUEBA:
    palabras = frase.split()
    rng.shuffle(palabras)
    desordenada = " ".join(palabras)
    print(f"{frase:<50} {perplejidad(frase):>7.2f} {perplejidad(desordenada):>12.2f} {perplejidad(frase, 0):>8}")
    assert perplejidad(frase) < perplejidad(desordenada)
    assert math.isclose(perplejidad(frase), perplejidad_producto(frase))
    assert perplejidad(frase, 0) == perplejidad_producto(frase, 0) == math.inf

frase                                                   PP  desordenada   PP k=0
la guagua sale del puerto a las seis de la tarde     22.71        56.30      inf
mi abuela prepara papas los sábados                  23.52        32.19      inf
el barco de la mañana llega a la playa               19.02        38.39      inf


## 4. Elegir $k$ con las frases de prueba

Laplace suele repartir demasiada probabilidad a lo no visto. Calcular la perplejidad conjunta del conjunto de prueba (todos los bigramas de las tres frases juntos) para una rejilla de valores de $k$ entre 0,001 y 2, quedarse con el mejor y comprobarlo con `scipy.optimize.minimize_scalar` sobre $\log k$.

In [5]:
from scipy.optimize import minimize_scalar


def perplejidad_conjunto(frases, k):
    n = sum(len(con_marcas(f)) - 1 for f in frases)
    return math.exp(-sum(log_prob(f, k) for f in frases) / n)


rejilla = np.geomspace(0.001, 2, 400)
valores = [perplejidad_conjunto(PRUEBA, k) for k in rejilla]
k_rejilla = rejilla[int(np.argmin(valores))]
for k in (0.001, 0.01, 0.1, 1, 2):
    print(f"k = {k:<6} PP = {perplejidad_conjunto(PRUEBA, k):.2f}")

opt = minimize_scalar(lambda lk: perplejidad_conjunto(PRUEBA, math.exp(lk)), bounds=(math.log(0.001), math.log(2)), method="bounded")
k_opt = math.exp(opt.x)
print(f"Mejor k: {k_rejilla:.3f} con la rejilla, {k_opt:.3f} con minimize_scalar (PP = {opt.fun:.2f})")
assert abs(math.log(k_rejilla) - opt.x) < 0.05
assert opt.fun < perplejidad_conjunto(PRUEBA, 1)

k = 0.001  PP = 6.40
k = 0.01   PP = 5.31
k = 0.1    PP = 7.56
k = 1      PP = 21.55
k = 2      PP = 28.95
Mejor k: 0.011 con la rejilla, 0.011 con minimize_scalar (PP = 5.31)


## 5. Generar frases con una cadena de Markov

Generar frases eligiendo cada palabra según $P(b \mid a)$ de máxima verosimilitud, empezando en `<s>` y parando en `</s>`, con `random.Random(37)`. Mostrar cinco frases, señalar cuáles no están en el entrenamiento y comprobar que todos sus bigramas se han visto. Comprobar también que, al muestrear 20 000 veces la palabra que sigue a `la`, las frecuencias se parecen a las probabilidades del modelo.

In [6]:
def siguiente(a, rng):
    palabras, pesos = zip(*siguientes[a].items())
    return rng.choices(palabras, weights=pesos)[0]


def generar(rng, maximo=20):
    frase, actual = [], "<s>"
    while len(frase) < maximo:
        actual = siguiente(actual, rng)
        if actual == "</s>":
            break
        frase.append(actual)
    return " ".join(frase)


rng = random.Random(37)
for _ in range(5):
    frase = generar(rng)
    print(("   " if frase in FRASES else "*  ") + frase)
    assert all(bigramas[par] for par in pairwise(con_marcas(frase)))
print("(* = frase nueva)")

muestras = Counter(siguiente("la", rng) for _ in range(20_000))
for b, c in siguientes["la"].most_common():
    esperado = c / unigramas["la"]
    print(f"la → {b:<9} modelo {esperado:.3f}  muestreo {muestras[b] / 20_000:.3f}")
    assert abs(muestras[b] / 20_000 - esperado) < 0.015

*  mi abuela prepara papas con mojo los domingos
*  la tarde llega a las seis
*  en el puerto a las nueve
*  mi tío pesca en invierno subimos al mercado
*  la playa hace calor en verano vamos a la tarde tiene nieve en la playa hace calor en la playa
(* = frase nueva)
la → playa     modelo 0.400  muestreo 0.402
la → guagua    modelo 0.200  muestreo 0.198
la → tarde     modelo 0.200  muestreo 0.201
la → estación  modelo 0.067  muestreo 0.066
la → gomera    modelo 0.067  muestreo 0.067
la → mañana    modelo 0.067  muestreo 0.066


## 6. Naive Bayes multinomial a mano

Con las opiniones de restaurantes y alojamientos de abajo, entrenar un Naive Bayes multinomial: la probabilidad a priori de cada clase es su proporción y la de cada palabra en una clase es $\frac{c(w, \text{clase}) + 1}{\sum_{w'} c(w', \text{clase}) + V}$ (Laplace). Clasificar una opinión sumando el logaritmo del a priori y los logaritmos de sus palabras y eligiendo la clase con mayor total. Usar la matriz de recuentos de `CountVectorizer` y mostrar la clase predicha para cuatro opiniones nuevas.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

OPINIONES = [
    ("la comida deliciosa y el personal amable", "pos"),
    ("habitación limpia y tranquila, muy recomendable", "pos"),
    ("el servicio excelente y la comida deliciosa", "pos"),
    ("personal amable y atento, volveremos", "pos"),
    ("un hotel tranquilo y limpio, excelente", "pos"),
    ("las papas arrugadas deliciosas y el precio bueno", "pos"),
    ("el desayuno bueno y el camarero atento", "pos"),
    ("apartamento limpio y tranquilo, volveremos", "pos"),
    ("trato amable y servicio excelente", "pos"),
    ("muy recomendable, el pescado bueno y fresco", "pos"),
    ("la terraza tranquila y el personal atento", "pos"),
    ("todo limpio, comida buena y precio recomendable", "pos"),
    ("la comida fría y el personal antipático", "neg"),
    ("habitación sucia y con mucho ruido", "neg"),
    ("el servicio lento y la comida cara", "neg"),
    ("personal antipático y lento, no volveremos", "neg"),
    ("un hotel ruidoso y sucio, horrible", "neg"),
    ("las papas frías y el precio caro", "neg"),
    ("el desayuno escaso y el camarero lento", "neg"),
    ("apartamento sucio y con ruido, no volveremos", "neg"),
    ("trato horrible y servicio lento", "neg"),
    ("nada recomendable, el pescado frío y caro", "neg"),
    ("la terraza sucia y el personal antipático", "neg"),
    ("todo caro, comida fría y trato horrible", "neg"),
]
NUEVAS = [
    "el servicio fue lento y la comida fría",
    "una terraza tranquila y un desayuno delicioso",
    "habitación cómoda pero muy cara",
    "el mojo estaba bueno",
]

textos = [t for t, _ in OPINIONES]
y = np.array([c for _, c in OPINIONES])
vectorizador = CountVectorizer(analyzer=tokens)
X = vectorizador.fit_transform(textos).toarray()
CLASES = np.array(["neg", "pos"])


def entrenar_nb(X, y, alfa=1.0):
    log_priori = np.log(np.array([(y == c).mean() for c in CLASES]))
    recuentos = np.array([X[y == c].sum(axis=0) for c in CLASES]) + alfa
    log_verosimilitud = np.log(recuentos / recuentos.sum(axis=1, keepdims=True))
    return log_priori, log_verosimilitud


def puntuaciones_nb(modelo, X):
    log_priori, log_verosimilitud = modelo
    return X @ log_verosimilitud.T + log_priori


modelo = entrenar_nb(X, y)
X_nuevas = vectorizador.transform(NUEVAS).toarray()
predichas = CLASES[puntuaciones_nb(modelo, X_nuevas).argmax(axis=1)]
for texto, clase in zip(NUEVAS, predichas):
    print(f"{clase}  {texto}")
assert (CLASES[puntuaciones_nb(modelo, X).argmax(axis=1)] == y).all()

neg  el servicio fue lento y la comida fría
pos  una terraza tranquila y un desayuno delicioso
pos  habitación cómoda pero muy cara
pos  el mojo estaba bueno


## 7. Comparar con `MultinomialNB`

Entrenar `MultinomialNB(alpha=1)` con la misma matriz y comprobar que sus probabilidades a priori, sus log-probabilidades por palabra y las puntuaciones conjuntas (`predict_joint_log_proba`) coinciden con las nuestras. Convertir las puntuaciones en probabilidades con *softmax* (restando el máximo para evitar desbordamientos) y compararlas con `predict_proba`.

In [8]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB(alpha=1).fit(X, y)
assert list(nb.classes_) == list(CLASES)
assert np.allclose(nb.class_log_prior_, modelo[0])
assert np.allclose(nb.feature_log_prob_, modelo[1])

conjuntas = puntuaciones_nb(modelo, X_nuevas)
assert np.allclose(conjuntas, nb.predict_joint_log_proba(X_nuevas))

e = np.exp(conjuntas - conjuntas.max(axis=1, keepdims=True))
probabilidades = e / e.sum(axis=1, keepdims=True)
assert np.allclose(probabilidades, nb.predict_proba(X_nuevas))
for texto, p in zip(NUEVAS, probabilidades):
    print(f"P(pos) = {p[1]:.3f}  {texto}")

P(pos) = 0.069  el servicio fue lento y la comida fría
P(pos) = 0.764  una terraza tranquila y un desayuno delicioso
P(pos) = 0.611  habitación cómoda pero muy cara
P(pos) = 0.805  el mojo estaba bueno


## 8. Palabras más indicativas

En Naive Bayes, el peso de una palabra a favor de una clase es el logaritmo del cociente $\log \frac{P(w \mid \text{pos})}{P(w \mid \text{neg})}$. Mostrar las cinco palabras más positivas y las cinco más negativas. Comprobar que, para una opinión, la diferencia de puntuaciones entre clases es la diferencia de los a priori más la suma de los pesos de sus palabras.

In [9]:
pesos = modelo[1][1] - modelo[1][0]
palabras = vectorizador.get_feature_names_out()
orden = np.argsort(pesos)
print("Positivas:", [(palabras[j], round(float(pesos[j]), 2)) for j in orden[::-1][:5]])
print("Negativas:", [(palabras[j], round(float(pesos[j]), 2)) for j in orden[:5]])

for fila, punt in zip(X_nuevas, conjuntas):
    assert math.isclose(punt[1] - punt[0], modelo[0][1] - modelo[0][0] + fila @ pesos)

Positivas: [('atento', 1.4), ('amable', 1.4), ('excelente', 1.4), ('limpio', 1.4), ('bueno', 1.4)]
Negativas: [('lento', -1.59), ('antipático', -1.37), ('horrible', -1.37), ('caro', -1.37), ('con', -1.08)]


## 9. Validación dejando uno fuera

Con tan pocas opiniones, acertar en el entrenamiento no dice nada. Estimar la precisión con validación cruzada dejando uno fuera: entrenar con 23 opiniones (el vocabulario incluido) y predecir la restante, 24 veces. Hacerlo a mano y comprobarlo con `cross_val_score` y `LeaveOneOut` sobre un `Pipeline` de `CountVectorizer` y `MultinomialNB`.

In [10]:
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.pipeline import make_pipeline

aciertos = []
for i in range(len(OPINIONES)):
    entrenamiento = [j for j in range(len(OPINIONES)) if j != i]
    vec = CountVectorizer(analyzer=tokens).fit([textos[j] for j in entrenamiento])
    m = entrenar_nb(vec.transform([textos[j] for j in entrenamiento]).toarray(), y[entrenamiento])
    pred = CLASES[puntuaciones_nb(m, vec.transform([textos[i]]).toarray()).argmax()]
    aciertos.append(pred == y[i])
    if pred != y[i]:
        print(f"Fallo: {textos[i]!r} → {pred}")

tubo = make_pipeline(CountVectorizer(analyzer=tokens), MultinomialNB(alpha=1))
puntos = cross_val_score(tubo, np.array(textos, dtype=object), y, cv=LeaveOneOut())
print(f"Precisión dejando uno fuera: {np.mean(aciertos):.0%}")
assert np.array_equal(puntos, np.array(aciertos, dtype=float))

Fallo: 'las papas arrugadas deliciosas y el precio bueno' → neg
Fallo: 'las papas frías y el precio caro' → pos
Fallo: 'el desayuno escaso y el camarero lento' → pos
Fallo: 'nada recomendable, el pescado frío y caro' → pos
Precisión dejando uno fuera: 83%


## 10. Sentimiento con un léxico

Sin entrenar nada, asignar a cada palabra una polaridad de un léxico escrito a mano (de −2 a +2). Las negaciones (`no`, `nada`, `nadie`, `nunca`) invierten el signo de las tres palabras siguientes y `muy` multiplica por 1,5 la siguiente. Escribir `sentimiento(texto)` con un recorrido de los tokens y una segunda versión que primero calcule un factor por posición y después haga un producto escalar con `numpy`; comprobar que dan lo mismo en todas las opiniones. Comparar las clases del léxico (puntuación positiva → `pos`) con las etiquetas.

In [11]:
LEXICO = {
    "deliciosa": 2, "deliciosas": 2, "delicioso": 2, "cómoda": 1, "amable": 1, "limpia": 1, "limpio": 1, "tranquila": 1, "tranquilo": 1,
    "recomendable": 1, "excelente": 2, "atento": 1, "buena": 1, "bueno": 1, "fresco": 1, "volveremos": 1,
    "fría": -1, "frías": -1, "frío": -1, "antipático": -2, "sucia": -2, "sucio": -2, "ruido": -1, "ruidoso": -1,
    "lento": -1, "cara": -1, "caro": -1, "horrible": -2, "escaso": -1,
}
NEGACIONES = {"no", "nada", "nadie", "nunca"}


def sentimiento(texto):
    total, negadas, intensidad = 0.0, 0, 1.0
    for w in tokens(texto):
        total += LEXICO.get(w, 0) * intensidad * (-1 if negadas else 1)
        intensidad = 1.5 if w == "muy" else 1.0
        negadas = 3 if w in NEGACIONES else max(negadas - 1, 0)
    return total


def sentimiento_vectorial(texto):
    ws = tokens(texto)
    signo, intensidad = np.ones(len(ws)), np.ones(len(ws))
    for i, w in enumerate(ws):
        if w in NEGACIONES:
            signo[i + 1 : i + 4] = -1
        elif w == "muy":
            intensidad[i + 1 : i + 2] = 1.5
    return float(np.array([LEXICO.get(w, 0) for w in ws]) @ (signo * intensidad))


for texto in textos + NUEVAS + ["no estaba nada mal, muy recomendable"]:
    assert math.isclose(sentimiento(texto), sentimiento_vectorial(texto)), texto

clases_lexico = np.where([sentimiento(t) > 0 for t in textos], "pos", "neg")
print(f"Acuerdo del léxico con las etiquetas: {(clases_lexico == y).mean():.0%}")
for texto in NUEVAS:
    print(f"{sentimiento(texto):+5.1f}  {texto}")
assert sentimiento("el pescado no estaba fresco") < 0 < sentimiento("el pescado estaba fresco")
assert sentimiento("nada recomendable") < 0 < sentimiento("muy recomendable") == 1.5

Acuerdo del léxico con las etiquetas: 100%
 -2.0  el servicio fue lento y la comida fría
 +3.0  una terraza tranquila y un desayuno delicioso
 -0.5  habitación cómoda pero muy cara
 +1.0  el mojo estaba bueno


## 11. Candidatos a una distancia de edición

Para corregir una palabra, Norvig genera todas las cadenas a una edición de distancia: borrar una letra, transponer dos contiguas, sustituir una letra o insertar una, con el alfabeto español (tildes y `ñ` incluidas). Escribir `ediciones1(palabra)` y comprobar el número de candidatos con la fórmula $n + (n - 1) + 33n + 33(n + 1)$ antes de quitar duplicados, y que cada candidato está realmente a distancia 1 calculada con la distancia de Damerau-Levenshtein restringida (programación dinámica).

In [12]:
ALFABETO = "abcdefghijklmnñopqrstuvwxyzáéíóúü"


def ediciones1_lista(palabra):
    cortes = [(palabra[:i], palabra[i:]) for i in range(len(palabra) + 1)]
    borrados = [a + b[1:] for a, b in cortes if b]
    transposiciones = [a + b[1] + b[0] + b[2:] for a, b in cortes if len(b) > 1]
    sustituciones = [a + c + b[1:] for a, b in cortes if b for c in ALFABETO]
    inserciones = [a + c + b for a, b in cortes for c in ALFABETO]
    return borrados + transposiciones + sustituciones + inserciones


def ediciones1(palabra):
    return set(ediciones1_lista(palabra)) - {palabra}


def distancia(a, b):
    d = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        d[i][0] = i
    for j in range(len(b) + 1):
        d[0][j] = j
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            coste = a[i - 1] != b[j - 1]
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1, d[i - 1][j - 1] + coste)
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                d[i][j] = min(d[i][j], d[i - 2][j - 2] + 1)
    return d[len(a)][len(b)]


assert len(ALFABETO) == 33
for palabra in ["mojo", "guagua", "a"]:
    m = len(palabra)
    assert len(ediciones1_lista(palabra)) == m + (m - 1) + 33 * m + 33 * (m + 1)
    assert all(distancia(palabra, c) == 1 for c in ediciones1(palabra))
    print(f"{palabra!r}: {len(ediciones1(palabra))} candidatos distintos")
assert distancia("gofoi", "gofio") == 1 and distancia("playa", "palyas") == 2

'mojo': 296 candidatos distintos
'guagua': 428 candidatos distintos
'a': 98 candidatos distintos


## 12. Un corrector ortográfico

Con la frecuencia de las palabras de las frases y las opiniones como vocabulario, escribir `corregir(palabra)`: si es conocida la deja; si no, elige el candidato conocido más frecuente a distancia 1; si no hay, a distancia 2; y si tampoco, la deja igual. Corregir una lista de palabras mal escritas y comprobar cada corrección por fuerza bruta: debe estar a la distancia mínima de todo el vocabulario (si es como mucho 2) y ser la más frecuente entre las que empatan.

In [13]:
FRECUENCIAS = Counter(w for texto in FRASES + textos for w in tokens(texto))


def conocidas(palabras):
    return {w for w in palabras if w in FRECUENCIAS}


def corregir(palabra):
    candidatos = (
        conocidas([palabra])
        or conocidas(ediciones1(palabra))
        or conocidas(e2 for e1 in ediciones1(palabra) for e2 in ediciones1(e1))
        or {palabra}
    )
    return max(sorted(candidatos), key=FRECUENCIAS.__getitem__)


def corregir_fuerza_bruta(palabra):
    distancias = {w: distancia(palabra, w) for w in FRECUENCIAS}
    minima = min(distancias.values())
    if minima > 2:
        return palabra
    empatadas = sorted(w for w, d in distancias.items() if d == minima)
    return max(empatadas, key=FRECUENCIAS.__getitem__)


for palabra in ["guaga", "plaia", "gofoi", "domigos", "pescdo", "teyde", "mojo", "xilófono"]:
    correccion = corregir(palabra)
    print(f"{palabra:<10} → {correccion}")
    assert correccion == corregir_fuerza_bruta(palabra)

guaga      → guagua
plaia      → playa
gofoi      → gofio
domigos    → domingos
pescdo     → pescado
teyde      → teide
mojo       → mojo
xilófono   → xilófono
